In [ ]:
flores={
    "tipo":["margarita","azucena","rosa"],
    "cant":[10,20,30],
    "precios":[1200,3000,1200]
}

In [ ]:
print("Bienvenido a mi floristería")
print("Se ofrecen los siguientes productos: \n")

# Encabezados de la tabla con alineación
print(f"{'Flores':<15} | {'Cantidad':<10} | {'Precio Unitario':<15}")
print("-" * 46)

# Recorremos las tres listas simultáneamente usando zip
for tipo, cantidad, precio in zip(flores['tipo'], flores['cant'], flores['precios']):
    print(f"{tipo.capitalize():<15} | {cantidad:<10} | ${precio:<14}")

In [ ]:
# Inicializamos el carrito de compras fuera del ciclo para que no se borre al repetir el menú
compra_user = {
    'tipo': [],
    'cantidad': [],
    'precio': []
}


# Interés aplicado cuando se paga con tarjeta de crédito en más de 1 cuota (5% sobre el total)
INTERES_CREDITO = 0.05
CUOTAS_PERMITIDAS = (1, 3, 6, 12, 24, 48)

# Registro simulado de tarjetas: número -> titular, tipo y saldo (en crédito, el saldo es el cupo disponible)
tarjetas = {
    "1111222233334444": {"titular": "maria lopez", "tipo": "credito", "saldo": 500000},
    "5555666677778888": {"titular": "carlos gomez", "tipo": "debito", "saldo": 100000},
}


def menu():
    """
    Muestra en pantalla el menú principal de la floristería con las opciones disponibles.
    Parametros:
    Ninguno
    """
    print("\nQué desea hacer?")
    print("1. Comprar flores")
    print("2. Pagar")
    print("3. Generar factura")
    print("4. Salir")


def calcular_total(carrito):
    """
    Calcula el total a pagar multiplicando la cantidad de cada flor del carrito por su precio unitario.
    Parametros:
    carrito : dict
    carrito = Diccionario con las listas 'tipo', 'cantidad' y 'precio' de las flores que el usuario ha comprado
    """
    total = 0
    for i in range(len(carrito["tipo"])):
        total += carrito["cantidad"][i] * carrito["precio"][i]
    return total


def mostrar_carrito(carrito):
    """
    Imprime en pantalla las flores que el usuario tiene actualmente en el carrito.
    Parametros:
    carrito : dict
    carrito = Diccionario con las listas 'tipo', 'cantidad' y 'precio' de las flores que el usuario ha comprado
    """
    for i, flor in enumerate(carrito['tipo']):
        print(f"{i+1}. {flor.capitalize()} - cantidad: {carrito['cantidad'][i]} - precio: ${carrito['precio'][i]}")


def comprar_flores(inventario, carrito):
    """
    Opción 1 del menú. Muestra las flores disponibles, permite quitar flores del carrito
    (devolviéndolas al inventario) y agrega al carrito la flor y la cantidad que el usuario elija,
    descontándola del inventario.
    Parametros:
    inventario : dict
    inventario = Diccionario con las listas 'tipo', 'cant' y 'precios' de las flores disponibles en la tienda
    carrito : dict
    carrito = Diccionario con las listas 'tipo', 'cantidad' y 'precio' de las flores que el usuario ha comprado
    """
    print("Flores disponibles: \n")
    for i, flor in enumerate(inventario['tipo']):
        print(f"{i+1}. {flor.capitalize()} - cantidad: {inventario['cant'][i]} - precio: ${inventario['precios'][i]}")

    # Qué pasa si el usuario entra y ya no hay flores disponibles
    if len(inventario['tipo']) == 0:
        print("No hay flores en stock, sorry :(")

    # Quitar del carrito si compré lo que no quería
    if len(carrito['tipo']) > 0:
        print("Flores en el carrito: \n")
        mostrar_carrito(carrito)

        print("Desea quitar o retirar alguna flor del carrito? (si/no)")
        quitar_opc = input().lower().strip()
        if quitar_opc == "si":
            print("Flores en el carrito: \n")
            mostrar_carrito(carrito)
            quitar_idx = int(input("Coloque el numero de la flor que deseas quitar del carrito: "))
            if quitar_idx > 0 and quitar_idx <= len(carrito['tipo']):
                flor_a_devolver = carrito['tipo'].pop(quitar_idx - 1)
                cantidad_a_devolver = carrito['cantidad'].pop(quitar_idx - 1)
                precio_a_devolver = carrito['precio'].pop(quitar_idx - 1)

                # Devolvemos la flor al inventario (si se había agotado, la volvemos a crear)
                if flor_a_devolver in inventario['tipo']:
                    idx_inventario = inventario['tipo'].index(flor_a_devolver)
                    inventario['cant'][idx_inventario] += cantidad_a_devolver
                else:
                    inventario['tipo'].append(flor_a_devolver)
                    inventario['cant'].append(cantidad_a_devolver)
                    inventario['precios'].append(precio_a_devolver)

    print("Ha seleccionado: Comprar flores\n")
    flor_opc = input("Seleccione la flor que desea comprar: ").lower().strip()
    if flor_opc in inventario['tipo']:
        idx = inventario['tipo'].index(flor_opc)
        flor_cant = int(input("Ingrese la cantidad que desea comprar: "))

        if flor_cant <= inventario['cant'][idx]:
            # Agregamos los valores a las listas del carrito
            carrito['tipo'].append(flor_opc)
            carrito['cantidad'].append(flor_cant)
            carrito['precio'].append(inventario['precios'][idx])

            # Restamos del inventario
            inventario['cant'][idx] -= flor_cant
            print(f"¡Se han añadido {flor_cant} {flor_opc}(s) al carrito!")

            # Removemos del inventario las flores si llegan a 0 usando .pop()
            if inventario['cant'][idx] == 0:
                inventario["tipo"].pop(idx)
                inventario["cant"].pop(idx)
                inventario["precios"].pop(idx)
        else:
            print("Cantidad no disponible")
    else:
        print(f"La flor {flor_opc} no está disponible")


def validar_tarjeta(registro, numero, titular, tipo):
    """
    Busca la tarjeta en el registro y verifica que el titular y el tipo (credito o debito) coincidan.
    Devuelve el diccionario de la tarjeta si es válida, o None si no lo es.
    Parametros:
    registro : dict
    registro = Diccionario con las tarjetas registradas (número -> titular, tipo y saldo)
    numero : str
    numero = Número de la tarjeta ingresado por el usuario
    titular : str
    titular = Nombre del titular ingresado por el usuario
    tipo : str
    tipo = Tipo de tarjeta que se desea usar, "credito" o "debito"
    """
    tarjeta = registro.get(numero)
    if tarjeta is None:
        return None
    if tarjeta["titular"] != titular.lower().strip() or tarjeta["tipo"] != tipo:
        return None
    return tarjeta


def pedir_cuotas():
    """
    Pide al usuario el número de cuotas hasta que ingrese una opción válida (1, 3, 6, 12, 24 o 48).
    Parametros:
    Ninguno
    """
    while True:
        try:
            cuotas = int(input("a cuantas cuotas se va a endeudar? (1,3,6,12,24,48): "))
        except ValueError:
            print("Ingrese un número válido.")
            continue
        if cuotas in CUOTAS_PERMITIDAS:
            return cuotas
        print("Número de cuotas no permitido.")


def calcular_total_con_interes(total, cuotas, interes):
    """
    Calcula el total a pagar con tarjeta de crédito. Con 1 cuota no se cobra interés;
    con más de 1 cuota se suma el porcentaje de interés sobre el total.
    Parametros:
    total : float
    total = Valor de la compra sin interés
    cuotas : int
    cuotas = Número de cuotas en las que se pagará la compra
    interes : float
    interes = Porcentaje de interés en formato decimal (0.05 equivale a 5%)
    """
    if cuotas == 1:
        return total
    return total * (1 + interes)


def pagar(carrito, registro_tarjetas):
    """
    Opción 2 del menú. Pide el medio de pago (efectivo, tarjeta de crédito, tarjeta de débito u otro),
    realiza los cálculos correspondientes (vuelto en efectivo; en tarjeta valida los datos y el saldo,
    y en crédito calcula el interés y el valor de cada cuota) y muestra el total a pagar por la compra.
    Parametros:
    carrito : dict
    carrito = Diccionario con las listas 'tipo', 'cantidad' y 'precio' de las flores que el usuario ha comprado
    registro_tarjetas : dict
    registro_tarjetas = Diccionario con las tarjetas registradas (número -> titular, tipo y saldo)
    """
    if len(carrito["tipo"]) == 0:
        print("El carrito de compras está vacío. ")
        return

    print("Seleccione el medio de pago: \n")
    print("1. Efectivo")
    print("2. Tarjeta de crédito")
    print("3. Tarjeta de débito")
    print("4. Otro medio de pago")
    pago_opc = input("Seleccione su opción: \n")

    total_pagar = calcular_total(carrito)

    # Si es efectivo hay que dar vuelto
    if pago_opc == "1":
        efectivo_recibido = float(input("Ingrese la cantidad o monto del dinero registrado porfi: "))
        if efectivo_recibido >= total_pagar:
            print(f"Su vuelto es: ${efectivo_recibido - total_pagar}")
        else:
            print("No tiene suficiente dinero para realizar la compra.")
            return

    # Si es con tarjeta hay que registrar número y titular, y revisar que haya saldo suficiente
    elif pago_opc == "2" or pago_opc == "3":
        tipo = "credito" if pago_opc == "2" else "debito"
        tarjeta_numero = input("Ingrese el número de su tarjeta: ").strip()
        tarjeta_titular = input("Ingrese el nombre del titular de la tarjeta porfi: ")

        tarjeta = validar_tarjeta(registro_tarjetas, tarjeta_numero, tarjeta_titular, tipo)
        if tarjeta is None:
            print("Tarjeta no registrada, titular incorrecto o tipo de tarjeta equivocado. No se puede realizar la compra.")
            return

        if pago_opc == "2":
            print("Ha seleccionado: Tarjeta de crédito")
            cuotas = pedir_cuotas()
            total_con_interes = calcular_total_con_interes(total_pagar, cuotas, INTERES_CREDITO)

            if total_con_interes > tarjeta["saldo"]:
                print(f"Cupo insuficiente. Total con interés: ${total_con_interes:.2f} - Cupo disponible: ${tarjeta['saldo']}")
                return

            tarjeta["saldo"] -= total_con_interes
            print(f"Interés aplicado: ${total_con_interes - total_pagar:.2f}")
            print(f"Pagará {cuotas} cuota(s) de ${total_con_interes / cuotas:.2f}")
            total_pagar = total_con_interes
        else:
            print("Ha seleccionado: Tarjeta de débito")
            if total_pagar > tarjeta["saldo"]:
                print(f"Saldo insuficiente. Total: ${total_pagar} - Saldo disponible: ${tarjeta['saldo']}")
                return
            tarjeta["saldo"] -= total_pagar

    print(f"\nUsted ha seleccionado la opción de pago {pago_opc}.")
    print(f"El total a pagar por su compra es: ${total_pagar:.2f}")


def generar_factura(carrito):
    """
    Opción 3 del menú. Imprime la factura con cada flor comprada, su cantidad, precio unitario y subtotal.
    Parametros:
    carrito : dict
    carrito = Diccionario con las listas 'tipo', 'cantidad' y 'precio' de las flores que el usuario ha comprado
    """
    print("Ha seleccionado: Generar factura\n")
    # Encabezados de la tabla con alineación
    print(f"{'Flores':<15} | {'Cantidad':<10} | {'Precio Unitario':<15} | {'Subtotal':<15}")
    print("-" * 64)
    for tipo, cantidad, precio in zip(carrito['tipo'], carrito['cantidad'], carrito['precio']):
        print(f"{tipo.capitalize():<15} | {cantidad:<10} | ${precio:<14} | ${precio*cantidad:<14}")


def salir():
    """
    Opción 4 del menú. Muestra el mensaje de que el usuario ha seleccionado salir.
    Parametros:
    Ninguno
    """
    print("Ha seleccionado: Salir\n")


def opcion_invalida():
    """
    Caso por defecto del menú. Avisa al usuario que el número ingresado no corresponde a ninguna opción.
    Parametros:
    Ninguno
    """
    print("Número incorrecto, por favor seleccione uno correcto")


while True:
    menu()

    opc = input("Seleccione su opción: \n")
    match opc:
        case "1":
            comprar_flores(flores, compra_user)
        case "2":
            pagar(compra_user, tarjetas)
        case "3":
            generar_factura(compra_user)
            break
        case "4":
            salir()
            break
        case _:
            opcion_invalida()

print("Gracias por emplear nuestros servicios. Feliz día")